In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import bce # type: ignore
import linear # type: ignore
import tanh # type: ignore
from per_lin_sig_bce import Per_Lin_Sig_BCE_Gradient # type: ignore
from common import repeat, test_function_checkgrad_2, test_module_compare_2, test_model_sgd_2 # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_Tanh_BCE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
    
    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear =  nn.Linear(in_features, out_features)
        self.tanh = nn.Tanh()

    def forward(self, x):
        z = self.model(x)
        return z
    
    def model(self, x):
        z = self.linear(x)
        t = self.tanh(z)
        p = 0.5 * (t + 1)       
        return p

    def loss(self, p, t):
        L = nn.BCELoss(reduction='mean')(p, t)
        return L

    def predict(self, x):
        with tr.no_grad():
            y = (self.model(x) > 0.5).float()
            return y

In [ ]:

class Per_Lin_Tanh_BCE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features, out_features, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Tanh + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Tanh] + [BinaryCrossEntropy]
        #

        self.linear = linear.Linear(in_features, out_features, W, b)
        self.tanh = tanh.Tanh()

    def model(self, x):
        z = self.linear(x)
        t = self.tanh(z)
        p = 0.5 * (t + 1)
        return p
    
    def loss(self, p, t):
        L = bce.BCE(reduction='mean')(p, t)
        return L

    def predict(self, x):
        with tr.no_grad():
            y = (self.model(x) > 0.5).float()
            return y
        
    def forward(self, x):
        p = self.model(x)
        return p

$$ z= xW + b $$

$$ \frac{\partial z}{\partial W} = x $$
$$ \frac{\partial z}{\partial b} = 1 $$

$$ \diamonds \diamonds \diamonds $$

$$ t = \tanh(z) \quad t \in (-1, \, +1) $$
$$ p(z) = 0.5(t(z) + 1) \quad p \in (0, \, 1) $$
$$ \frac {\partial p}{\partial z} = 0.5 (1 - t^2) $$

$$ \diamonds \diamonds \diamonds $$

$$ L = -y \log(p) - (1-y) \log(1-p) $$
$$ \frac{\partial L}{\partial p} = \frac{p-y}{p(1-p)} $$
$$ \frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} = 2(p-y) $$
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial W} = x \cdot 2(p-y) $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial b} = 2(p-y) $$

$$ \diamonds \diamonds \diamonds $$

In [ ]:
class Per_Lin_Tanh_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def __model(x, W, b):
        z = linear.linear(x, W, b)
        t = tanh.tanh(z)
        p = 0.5 * (t + 1)
        return p
    
    @staticmethod
    def __loss(p, t):
        L = bce.bce(p, t)
        return L

    @staticmethod
    def predict(x, W, b):
        with tr.no_grad():
            y = (Per_Lin_Tanh_BCE_Gradient_Function.__model(x, W, b) >= 0.5).float()
            return y

    @staticmethod
    def forward(ctx, x, W, b, t):
        p = Per_Lin_Tanh_BCE_Gradient_Function.__model(x, W, b)
        L = Per_Lin_Tanh_BCE_Gradient_Function.__loss(p, t)

        ctx.save_for_backward(x, W, p, t)

        return L
    
    @staticmethod
    def backward(ctx, output_grad):
        x = ctx.saved_tensors[0]
        W = ctx.saved_tensors[1]
        p = ctx.saved_tensors[2]
        t = ctx.saved_tensors[3]

        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out
        
        dL_dz = 2 * (p - t)

        # Average over all elements
        N = S * FO
        dL_dz = dL_dz / N

        # For this example output_grad == 1
        dL_dz = dL_dz * output_grad
        assert dL_dz.shape == (S, FO)
        
        # (FI, FO) = (S, FI).T @ (S, FO)
        dL_dW = x.t() @ dL_dz
        assert dL_dW.shape == (FI, FO)

        dL_db = dL_dz.sum(dim=0)
        assert dL_db.shape == (FO,)

        return (None, dL_dW, dL_db, None)
    

class Per_Lin_Tanh_BCE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # In general the perceptron model is separable from the loss function, 
    # but since the total gradient is calculated manually, the loss function 
    # is an integrated part of it. This is helper for testing to indicate that 
    # the `forward` method expects both, `x` and `y`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `weight` has shape `(out_features, in_features)` to be `nn.Linear` compatible
        if W is None:
            self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_Tanh_BCE_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            y = Per_Lin_Tanh_BCE_Gradient_Function.__model(x, self.weight.t(), self.bias)
            return y
    
    def loss(self, p, t):
        with tr.no_grad():
            L = Per_Lin_Tanh_BCE_Gradient_Function.__loss(p, t)
            return L

    def predict(self, x):
        with tr.no_grad():
            y = Per_Lin_Tanh_BCE_Gradient_Function.predict(x, self.weight.t(), self.bias)
            return y
        
    def forward(self, x, t):
        y = Per_Lin_Tanh_BCE_Gradient_Function.apply(x, self.weight.t(), self.bias, t)
        return y

In [ ]:
def test_logical_fn(model, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    y = bool_fn(x[:, 0], x[:, 1]).float().unsqueeze(1)
    return test_model_sgd_2(model, x, y, epochs, lr=lr)


if __name__ == "__main__":
    test_function_checkgrad_2(Per_Lin_Tanh_BCE_Gradient_Function, 1, 1, 1, prec=0.01)
    test_function_checkgrad_2(Per_Lin_Tanh_BCE_Gradient_Function, 2, 2, 2, prec=0.01)
    test_function_checkgrad_2(Per_Lin_Tanh_BCE_Gradient_Function, 3, 3, 3, prec=0.01)

    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 1, 1, 1)
    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 10, 1, 1)
    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 10, 20, 1)
    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 10, 20, 30)

    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 1, 1, 1)
    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 10, 1, 1)
    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 10, 20, 1)
    test_module_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 10, 20, 30)

    # Tanh keeps gradients alive much longer than sigmoid, so linear+tanh+BCE converges faster on logical functions.

    for epochs in [100, 200, 300, 400, 500]:
        print(f"{epochs}/OR/Tanh: {repeat(lambda: test_logical_fn(Per_Lin_Tanh_BCE_Gradient(2, 1), tr.logical_or, epochs=epochs)):.2f}")
        print(f"{epochs}/OR/Sig : {repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), tr.logical_or, epochs=epochs)):.2f}\n")

    # # The perceptron Linear + Tanh + BCE is still linear, and eventhough it converges faster than sigmoid, it still cannot learn the XOR function.

    for epochs in [500, 1000, 1500]:
        print(f"{epochs}/XOR/Tanh: {repeat(lambda: test_logical_fn(Per_Lin_Tanh_BCE_Gradient(2, 1), tr.logical_xor, epochs=epochs)):.2f}")
        print(f"{epochs}/XOR/Sig : {repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), tr.logical_xor, epochs=epochs)):.2f}\n")